# 03 - Apache Iceberg

Este notebook cria uma tabela `Apache Iceberg` usando um catalogo local do tipo `hadoop`. Assim como no notebook de Delta, o objetivo principal e demonstrar `INSERT`, `UPDATE` e `DELETE` dentro do Apache Spark.


In [ ]:
import os
from pathlib import Path
import shutil

from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_HADOOP = PROJECT_ROOT / "windows-hadoop"
WINUTILS = LOCAL_HADOOP / "bin" / "winutils.exe"
if WINUTILS.exists():
    os.environ["HADOOP_HOME"] = str(LOCAL_HADOOP)
    os.environ["hadoop.home.dir"] = str(LOCAL_HADOOP)
else:
    print("Aviso: winutils.exe nao encontrado. Em Windows, coloque o arquivo em windows-hadoop/bin antes de executar o notebook.")

DATA_PATH = PROJECT_ROOT / "data" / "vendas.csv"
ICEBERG_WAREHOUSE = PROJECT_ROOT / "warehouse" / "iceberg"

shutil.rmtree(ICEBERG_WAREHOUSE, ignore_errors=True)
ICEBERG_WAREHOUSE.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("iceberg-demo")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.1")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", ICEBERG_WAREHOUSE.as_posix())
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [ ]:
vendas_df = (
    spark.read
    .option("header", True)
    .csv(str(DATA_PATH))
    .withColumn("id_venda", F.col("id_venda").cast("bigint"))
    .withColumn("id_cliente", F.col("id_cliente").cast("int"))
    .withColumn("id_produto", F.col("id_produto").cast("int"))
    .withColumn("data_venda", F.to_date("data_venda"))
    .withColumn("quantidade", F.col("quantidade").cast("int"))
    .withColumn("preco_unitario", F.col("preco_unitario").cast("double"))
)

vendas_df.createOrReplaceTempView("staging_vendas")
vendas_df.orderBy("id_venda").show(truncate=False)


In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.loja")
spark.sql("DROP TABLE IF EXISTS local.loja.vendas_iceberg")

spark.sql(
    """
    CREATE TABLE local.loja.vendas_iceberg
    USING iceberg
    AS SELECT * FROM staging_vendas
    """
)

spark.sql(
    "SELECT id_venda, nome_cliente, nome_produto, quantidade, status_pagamento FROM local.loja.vendas_iceberg ORDER BY id_venda"
).show(truncate=False)


## INSERT em Apache Iceberg

Abaixo, um novo registro e inserido na tabela Iceberg.


In [ ]:
spark.sql(
    """
    INSERT INTO local.loja.vendas_iceberg
    VALUES (
        1011,
        6,
        'Marcos Lima',
        'Criciuma',
        106,
        'Mouse Gamer',
        'Perifericos',
        DATE '2026-03-11',
        2,
        180.00,
        'pago'
    )
    """
)

spark.sql("SELECT * FROM local.loja.vendas_iceberg WHERE id_venda = 1011").show(truncate=False)


## UPDATE em Apache Iceberg

O exemplo abaixo altera o status de pagamento de uma venda existente.


In [ ]:
spark.sql(
    """
    UPDATE local.loja.vendas_iceberg
    SET status_pagamento = 'em_analise'
    WHERE id_venda = 1004
    """
)

spark.sql("SELECT * FROM local.loja.vendas_iceberg WHERE id_venda = 1004").show(truncate=False)


## DELETE em Apache Iceberg

O registro cancelado e removido da tabela.


In [ ]:
spark.sql(
    """
    DELETE FROM local.loja.vendas_iceberg
    WHERE id_venda = 1008
    """
)

spark.sql("SELECT * FROM local.loja.vendas_iceberg WHERE id_venda = 1008").show(truncate=False)


In [ ]:
spark.sql(
    "SELECT id_venda, nome_cliente, nome_produto, quantidade, status_pagamento FROM local.loja.vendas_iceberg ORDER BY id_venda"
).show(truncate=False)

spark.sql(
    "SELECT committed_at, operation, snapshot_id FROM local.loja.vendas_iceberg.snapshots ORDER BY committed_at"
).show(truncate=False)


In [ ]:
spark.stop()
